# KDD Cup 99 - Model Comparison

Comparative analysis of all models (Isolation Forest, SGD, MLP, Ensemble) on the KDD dataset.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load evaluation results
iso_df = pd.read_csv('../../evaluation/isolation_forest/kdd/isolation_forest_evaluation.csv')
sgd_df = pd.read_csv('../../evaluation/sgd/kdd/sgd_evaluation.csv')
mlp_df = pd.read_csv('../../evaluation/mlp/kdd/mlp_evaluation.csv')
ensemble_df = pd.read_csv('../../evaluation/ensemble/kdd/ensemble_evaluation.csv')

print('Evaluation data loaded successfully')
print(f'Isolation Forest: {len(iso_df)} row(s)')
print(f'SGD: {len(sgd_df)} row(s)')
print(f'MLP: {len(mlp_df)} row(s)')
print(f'Ensemble: {len(ensemble_df)} row(s)')

## 1. Results Summary Table

In [ ]:
# Build comparison table
iso_row = iso_df.iloc[0]
sgd_row = sgd_df.iloc[0]
mlp_row = mlp_df.iloc[0]
ensemble_row = ensemble_df.iloc[0]

comparison = pd.DataFrame({
    'Model': ['Isolation Forest', 'SGD', 'MLP', 'Ensemble'],
    'Accuracy': [iso_row['accuracy'], sgd_row['accuracy'], mlp_row['accuracy'], ensemble_row['accuracy']],
    'Balanced Acc.': ['-', sgd_row['balanced_accuracy'], mlp_row['balanced_accuracy'], ensemble_row['balanced_accuracy']],
    'Precision (Anomaly)': [iso_row['precision_anomaly'], sgd_row['precision_anomaly'], mlp_row['precision_anomaly'], ensemble_row['precision_anomaly']],
    'Recall (Anomaly)': [iso_row['recall_anomaly'], sgd_row['recall_anomaly'], mlp_row['recall_anomaly'], ensemble_row['recall_anomaly']],
    'F1 (Anomaly)': [iso_row['f1_anomaly'], sgd_row['f1_anomaly'], mlp_row['f1_anomaly'], ensemble_row['f1_anomaly']],
    'F1 Macro': [iso_row['f1_macro'], sgd_row['f1_macro'], mlp_row['f1_macro'], ensemble_row['f1_macro']],
    'Stage 2 Call Rate': ['-', sgd_row['stage2_call_rate'], mlp_row['stage2_call_rate'], ensemble_row['stage2_call_rate']],
    'Time (s)': [iso_row['elapsed_sec'], sgd_row['total_time_sec'], mlp_row['total_time_sec'], ensemble_row['total_time_sec']],
})

comparison

## 2. Performance Metrics Comparison

In [ ]:
# Stage 2 models only (exclude Isolation Forest for fair comparison)
models = ['SGD', 'MLP', 'Ensemble']
colors = {'SGD': '#1f77b4', 'MLP': '#ff7f0e', 'Ensemble': '#2ca02c'}

metrics = {
    'Accuracy': [sgd_row['accuracy'], mlp_row['accuracy'], ensemble_row['accuracy']],
    'Balanced Accuracy': [sgd_row['balanced_accuracy'], mlp_row['balanced_accuracy'], ensemble_row['balanced_accuracy']],
    'F1 Macro': [sgd_row['f1_macro'], mlp_row['f1_macro'], ensemble_row['f1_macro']],
    'Precision (Anomaly)': [sgd_row['precision_anomaly'], mlp_row['precision_anomaly'], ensemble_row['precision_anomaly']],
    'Recall (Anomaly)': [sgd_row['recall_anomaly'], mlp_row['recall_anomaly'], ensemble_row['recall_anomaly']],
    'F1 (Anomaly)': [sgd_row['f1_anomaly'], mlp_row['f1_anomaly'], ensemble_row['f1_anomaly']],
}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('KDD Cup 99 - Stage 2 Model Comparison', fontsize=16, fontweight='bold')

for idx, (metric_name, values) in enumerate(metrics.items()):
    ax = axes[idx // 3, idx % 3]
    bars = ax.bar(models, values, color=[colors[m] for m in models], alpha=0.8, edgecolor='black')
    ax.set_title(metric_name, fontsize=12, fontweight='bold')
    ax.set_ylim([min(values) - 0.05, 1.0])
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    # Add value labels on bars
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('kdd_model_comparison_metrics.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Accuracy vs Processing Time Trade-off

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

times = [sgd_row['total_time_sec'], mlp_row['total_time_sec'], ensemble_row['total_time_sec']]
accs = [sgd_row['accuracy'], mlp_row['accuracy'], ensemble_row['accuracy']]
f1s = [sgd_row['f1_macro'], mlp_row['f1_macro'], ensemble_row['f1_macro']]

# Accuracy vs Time
ax = axes[0]
for m, t, a in zip(models, times, accs):
    ax.scatter(t, a, s=300, color=colors[m], alpha=0.8, edgecolors='black', linewidth=2, zorder=5)
    ax.annotate(f'{m}\n{a:.4f}', xy=(t, a), xytext=(10, 10),
               textcoords='offset points', fontsize=9, fontweight='bold',
               bbox=dict(boxstyle='round,pad=0.4', facecolor=colors[m], alpha=0.3))
ax.set_xlabel('Processing Time (s)', fontsize=11, fontweight='bold')
ax.set_ylabel('Accuracy', fontsize=11, fontweight='bold')
ax.set_title('Accuracy vs Processing Time', fontsize=12, fontweight='bold')
ax.grid(alpha=0.3, linestyle='--')

# F1 Macro vs Time
ax = axes[1]
for m, t, f in zip(models, times, f1s):
    ax.scatter(t, f, s=300, color=colors[m], alpha=0.8, edgecolors='black', linewidth=2, zorder=5)
    ax.annotate(f'{m}\n{f:.4f}', xy=(t, f), xytext=(10, 10),
               textcoords='offset points', fontsize=9, fontweight='bold',
               bbox=dict(boxstyle='round,pad=0.4', facecolor=colors[m], alpha=0.3))
ax.set_xlabel('Processing Time (s)', fontsize=11, fontweight='bold')
ax.set_ylabel('F1 Macro', fontsize=11, fontweight='bold')
ax.set_title('F1 Macro vs Processing Time', fontsize=12, fontweight='bold')
ax.grid(alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('kdd_time_tradeoff.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Confusion Matrix Analysis

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
fig.suptitle('KDD Cup 99 - Confusion Matrices', fontsize=14, fontweight='bold')

all_rows = [
    ('Isolation Forest', iso_row),
    ('SGD', sgd_row),
    ('MLP', mlp_row),
    ('Ensemble', ensemble_row),
]

for ax, (name, row) in zip(axes, all_rows):
    cm = np.array([[row['confusion_tn'], row['confusion_fp']],
                   [row['confusion_fn'], row['confusion_tp']]])
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Normal', 'Anomaly'],
                yticklabels=['Normal', 'Anomaly'])
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig('kdd_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Detailed Analysis

In [ ]:
print('=' * 90)
print('KDD CUP 99 - DETAILED MODEL COMPARISON')
print('=' * 90)

print(f'\nStage 2 Call Rate: {sgd_row["stage2_call_rate"]:.2%} of traffic reaches Stage 2')
print(f'Total test samples: {int(sgd_row["total_processed"]):,}')

print('\n--- Stage 2 Models ---')
for name, row in [('SGD', sgd_row), ('MLP', mlp_row), ('Ensemble', ensemble_row)]:
    print(f'\n  {name}:')
    print(f'    Accuracy:          {row["accuracy"]:.4f}')
    print(f'    Balanced Accuracy: {row["balanced_accuracy"]:.4f}')
    print(f'    F1 Macro:          {row["f1_macro"]:.4f}')
    print(f'    F1 Anomaly:        {row["f1_anomaly"]:.4f}')
    print(f'    Recall Anomaly:    {row["recall_anomaly"]:.4f}')
    print(f'    Time:              {row["total_time_sec"]:.2f}s')

print('\n--- Trade-off Analysis ---')
time_mlp_vs_sgd = ((mlp_row['total_time_sec'] - sgd_row['total_time_sec']) / sgd_row['total_time_sec'] * 100)
f1_mlp_vs_sgd = ((mlp_row['f1_macro'] - sgd_row['f1_macro']) / sgd_row['f1_macro'] * 100)
time_ens_vs_mlp = ((ensemble_row['total_time_sec'] - mlp_row['total_time_sec']) / mlp_row['total_time_sec'] * 100)
f1_ens_vs_mlp = ((ensemble_row['f1_macro'] - mlp_row['f1_macro']) / mlp_row['f1_macro'] * 100)

print(f'  MLP vs SGD:      {time_mlp_vs_sgd:+.1f}% time for {f1_mlp_vs_sgd:+.1f}% F1 Macro')
print(f'  Ensemble vs MLP: {time_ens_vs_mlp:+.1f}% time for {f1_ens_vs_mlp:+.1f}% F1 Macro')

# Best model
best_f1 = max(sgd_row['f1_macro'], mlp_row['f1_macro'], ensemble_row['f1_macro'])
best_name = ['SGD', 'MLP', 'Ensemble'][[sgd_row['f1_macro'], mlp_row['f1_macro'], ensemble_row['f1_macro']].index(best_f1)]
print(f'\n  Best model by F1 Macro: {best_name} ({best_f1:.4f})')
print('=' * 90)